<a href="https://colab.research.google.com/github/NajmehNayyer/Social_Status/blob/main/Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import numpy as np
import pandas as pd

In [6]:
file_id = '1mRPLoOGPVAwMmRT0zSrDCIq4dzQ4oAWO'
output_filename = 'data.csv'

!gdown --id {file_id} -O {output_filename}

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1mRPLoOGPVAwMmRT0zSrDCIq4dzQ4oAWO
To: /content/data.csv
100% 6.84k/6.84k [00:00<00:00, 23.5MB/s]


In [7]:
df = pd.read_csv("data.csv")
df.head()

,ID,gender,marriage,religious_believes,family_income,personal_income,abode,father_education,unnecessary_expenses,entertainment_expenses,...,volunteer_work,socialmedia,friends,science,politics,culture,religion,others,nothing,family
0,1,1,1,2,30,0.0,3,3,۵۰ الی ۷۰ درصد,۱ الی ۳ میلیون تومان,...,متوسط,۱ تا ۳ ساعت,متوسط,علمی,NaN,فرهنگی,NaN,NaN,NaN,1
1,2,1,1,3,60,0.0,4,2,۳۰ الی ۵۰ درصد,۱ الی ۳ میلیون تومان,...,خوب,۱ تا ۳ ساعت,متوسط,NaN,NaN,فرهنگی,NaN,NaN,NaN,2
2,3,1,1,3,15,0.0,2,1,۳۰ الی ۵۰ درصد,زیر ۱ میلیون تومان,...,ضعیف,بالای ۵ ساعت,زیاد,NaN,NaN,NaN,NaN,NaN,هیچکدام,1
3,4,1,1,2,17.5,0.0,3,1,۷۰ الی ۱۰۰ درصد,زیر ۱ میلیون تومان,...,ضعیف,۱ تا ۳ ساعت,متوسط,علمی,NaN,NaN,NaN,NaN,NaN,2
4,5,1,1,3,14,0.0,3,3,۵۰ الی ۷۰ درصد,زیر ۱ میلیون تومان,...,متوسط,۱ تا ۳ ساعت,زیاد,NaN,NaN,NaN,NaN,سایر,NaN,1


#Preprocessing

## Correcting Data Collector's Mistakes

**Replaceing values with new ones**

Here, I will correct some of the data collector's mistakes, so having a bigger number on the scale means better performance in that task. At the same time, I will also convert my strings to numerical values, especially since they are Persian.

In [8]:
# Replace values with numerical values
replaced_names = {
    'unnecessary_expenses':  {'۱۰ الی ۳۰ درصد': 1, '۳۰ الی ۵۰ درصد': 2, '۵۰ الی ۷۰ درصد': 3, '۷۰ الی ۱۰۰ درصد': 4},
    'entertainment_expenses': {'زیر ۱ میلیون تومان': 1, '۱ الی ۳ میلیون تومان': 2, '۳ الی ۵ میلیون تومان': 3, 'بالای ۵ میلیون تومان': 4},
    'reading_hours':   {'زیر یکساعت': 1, 'یک تا سه ساعت': 2, 'سه تا هفت': 3, 'بالای هفت ساعت...': 4},
    'sport':  {'خیر ، فاقد مهارت ورزشی': 1, 'در حد مبتدی': 2, 'در حد متوسط': 3, 'بله ، در حد عالی': 4},
    'english':    {'فاقد آشنایی': 1, 'در حد کم': 2, 'متوسط': 3, 'زیاد': 4, 'عالی': 5},
    'volunteer_work':   {'خیلی ضعیف': 1, 'ضعیف': 2, 'متوسط': 3, 'خوب': 4},
    'socialmedia': {'زیر ۱ ساعت': 1, '۱ تا ۳ ساعت': 2, '۳ تا ۵ ساعت': 3, 'بالای ۵ ساعت': 4},
    'friends':    {'خیلی کم': 1, 'کم': 2, 'متوسط': 3, 'زیاد': 4},
    'family': {1: 3, 3: 1},
    'abode':  {1: 5, 2: 4, 4: 2, 5: 1}}
df = df.replace(replaced_names)

/tmp/ipykernel_14970/935030523.py:13: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace(replaced_names)



**Combining and Dropping Some of The Columns**

There are six columns related to the type of NGOs individuals are involved in.

`NGO_Types = ['science', 'culture', 'religion', 'politics', 'others', 'nothing']`.
I will count involvement in each of these columns, except for `NGO_Types['nothing']`, and then create a column named "NGOs" for them.

The nothing column isn't needed here, so I will drop it alongside marriage status since none of the individuals is married in our survey.

In [9]:
# Count categories of NGOs they're involved in
NGOs = ['science', 'culture', 'religion', 'politics', 'others']
for NGO in NGOs: #fill NaNs with 0 and none NaNs with 1
  df[NGO] = df[NGO].fillna(0).apply(lambda x: 1 if x != 0 else 0)
df['ngo'] = df[NGOs].sum(axis=1)

# Drop columns that aren't needed
df.drop(['nothing', 'marriage'] + NGOs, axis=1, inplace=True)

## Handling Missing Data

After I turned all my data points to numeric values I search for missing data points and fill those with appropriate values.

Columns with missing data:
*  Family Income: Median due to that variability.
*  Person's Income: 0 because most of the individuals didn't make any money, so if they didn't answer this, they probably were the same.


In [10]:
# Convert columns to numerical values
df = df.apply(pd.to_numeric, errors='coerce')

# Fill missing values
df['family_income'] = df['family_income'].fillna(df['family_income'].median())
df['computer_skills'] = df['computer_skills'].fillna(df['computer_skills'].mean()).astype(int)

In [11]:
df['family_income_score'] = pd.cut(df['family_income'], bins=[10, 28, 46, 64, 82, 100], labels=[1, 2, 3, 4, 5], right=True, include_lowest=True)
df['personal_income_score'] = pd.cut(df['personal_income'], bins=[0, 5, 10, 50], labels=[1, 2, 3], right=True, include_lowest=True)

df['family_income_score'].astype(float)
df['personal_income_score'].astype(float)

,personal_income_score
0,1.0
1,1.0
2,1.0
3,1.0
4,1.0
5,1.0
6,1.0
7,1.0
8,1.0
9,2.0


## Converting All the Values to Scores

**Income varibles**: We ask people how much they make per month (million Rial) but those valuses need to be transfered to scores.

**Personal Income**: Only a minority had thier own outcome we have 3 scores for Personal income and 5 for Family Income.

In [12]:
# Create a dictionary for variables as label and sub-variables for their values
survey_variables= {
    'economical': {'family_income_score', 'personal_income_score', 'abode', 'unnecessary_expenses', 'entertainment_expenses', 'work_experience'},
    'cultural': {'religious_believes', 'father_education', 'reading_hours', 'sport', 'computer_skills', 'english'},
    'social': {'gender', 'volunteer_work', 'socialmedia', 'friends', 'ngo', 'family'}}

# Create columns for the main scores
for variable, subvariable in survey_variables.items():
  df[variable] = df[list(subvariable)].mean(axis=1)

## Creating Our Main Variables

In this project, we aim to measure the classes of economic, social, and cultural scores to compare each person's score to others.
For that, we should create economic, social, cultural, socio-cultural, and overall scores.

In [13]:
# Create Additional variables
df['sociocultural'] = df[['cultural', 'social']].mean(axis=1)
df['overall'] = df[['economical', 'cultural', 'social']].mean(axis=1)
add_variables = ['sociocultural', 'overall']

# Rearrange the data
new_order = ['ID', 'personal_income', 'family_income'] + [subvariable for variable in survey_variables.keys() for subvariable in list(survey_variables[variable])] + list(survey_variables.keys()) + add_variables

df = df[new_order]

In [14]:
# Write min and max based on survey options
min_max = {'gender':[1, 2], 'religious_believes':[1, 5], 'sport':[1, 4], 'abode':[1, 5], 'father_education':[1,5],'unnecessary_expenses':[1,4], 'entertainment_expenses':[1,4], 'work_experience':[1,3], 'reading_hours':[1,4], 'computer_skills':[1,4], 'english':[1,5], 'volunteer_work':[1,5],'socialmedia':[1,4], 'friends':[1,5], 'ngo':[0,5] , 'family_income_score':[1,5], 'personal_income_score':[1,3],  'family':[1,3], 'family_income': [10, 100], 'personal_income': [0, 50]}

## Calculate min and max

The theoretical minimum and maximum are different from the ones participants actually choose, so I calculated the theoretical ones for normalization.

In [15]:
def compute_min_max(*args):

    args_min_list = []; args_max_list = []

    for arg in args:

        # Get the names of subvaiables that create our variable
        needed_subvars = survey_variables.get(arg)

        # Get all the minimums and maximums of the above list
        min_list = [min_max.get(key)[0] for key in min_max.keys() if key in needed_subvars]
        max_list = [min_max.get(key)[1] for key in min_max.keys() if key in needed_subvars]

        # Append the numbers to our args lists
        args_min_list.extend(min_list)
        args_max_list.extend(max_list)

        return np.mean(args_min_list), np.mean(args_max_list)

In [16]:
for var in ["social", "cultural", "economical"]:
    min, max = compute_min_max(var)
    min_max[var] = [min, max]

socio_min, socio_max = compute_min_max("social", "cultural")
min_max["sociocultural"] = [socio_min, socio_max]

overall_min, overall_max = compute_min_max("social", "cultural", "economical")
min_max["overall"] = [overall_min, overall_max]

df.head()

,ID,personal_income,family_income,family_income_score,unnecessary_expenses,personal_income_score,abode,entertainment_expenses,work_experience,sport,...,ngo,family,gender,socialmedia,volunteer_work,economical,cultural,social,sociocultural,overall
0,1,0.0,30.0,2,3,1,3,2,1,3,...,2,3,1,2,3,2.000000,2.500000,2.333333,2.416667,2.277778
1,2,0.0,60.0,3,2,1,2,2,1,2,...,1,2,1,2,4,1.833333,2.000000,2.166667,2.083333,2.000000
2,3,0.0,15.0,1,2,1,4,1,3,2,...,0,3,1,4,2,2.000000,1.666667,2.333333,2.000000,2.000000
3,4,0.0,17.5,1,4,1,3,1,1,1,...,1,2,1,2,2,1.833333,2.333333,1.833333,2.083333,2.000000
4,5,0.0,14.0,1,3,1,3,1,1,2,...,1,3,1,2,3,1.666667,2.166667,2.333333,2.250000,2.055556


In [23]:
df.to_csv('preprocessed_social_status.csv', index=False)

# Normalization

In [18]:
ndf = df.copy()

for col in ndf.columns:
    if col != "ID":
        ndf[col] = pd.to_numeric(ndf[col], errors='coerce')
        ndf[col] = ndf[col]/min_max[col][1] * 100

In [19]:
# Reordering the New Data Frame
ndf_og = ndf.columns
ndf_new_order = ['ID', 'personal_income_score', 'family_income_score', 'entertainment_expenses', 'abode', 'work_experience', 'unnecessary_expenses', 'gender', 'english', 'sport', 'father_education', 'religious_believes', 'computer_skills', 'reading_hours', 'socialmedia', 'friends', 'ngo', 'family', 'volunteer_work', 'economical', 'cultural', 'social', 'sociocultural', 'overall']
ndf = ndf[ndf_new_order].round(4)

ndf.head()

,ID,personal_income_score,family_income_score,entertainment_expenses,abode,work_experience,unnecessary_expenses,gender,english,sport,...,socialmedia,friends,ngo,family,volunteer_work,economical,cultural,social,sociocultural,overall
0,1,33.3333,40.0,50.0,60.0,33.3333,75.0,50.0,60.0,75.0,...,50.0,60.0,40.0,100.0000,60.0,50.0000,55.5556,58.3333,60.4167,56.9444
1,2,33.3333,60.0,50.0,40.0,33.3333,50.0,50.0,60.0,50.0,...,50.0,60.0,20.0,66.6667,80.0,45.8333,44.4444,54.1667,52.0833,50.0000
2,3,33.3333,20.0,25.0,80.0,100.0000,50.0,50.0,40.0,50.0,...,100.0,80.0,0.0,100.0000,40.0,50.0000,37.0370,58.3333,50.0000,50.0000
3,4,33.3333,20.0,25.0,60.0,33.3333,100.0,50.0,60.0,25.0,...,50.0,60.0,20.0,66.6667,40.0,45.8333,51.8519,45.8333,52.0833,50.0000
4,5,33.3333,20.0,25.0,60.0,33.3333,75.0,50.0,60.0,50.0,...,50.0,80.0,20.0,100.0000,60.0,41.6667,48.1481,58.3333,56.2500,51.3889


In [20]:
if len(df.columns) == len(ndf_og): print('All the columns are transformed and transferred to new data.')

All the columns are transformed and transferred to new data.


In [22]:
ndf.to_csv('normalized_social_status', index=False)